In [1]:
import pyodbc
import pandas as pd

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

In [2]:
euro_govs = pd.read_csv('gov_fut.csv')

In [3]:
euro_govs_isin = tuple(euro_govs['isin'].unique())

In [ ]:
# sample: Sep 2024 Bund future (FGBL) on one trading day
sample_isin = 'DE000F0FSQN3'
sample_date = '2024-07-15'

In [ ]:
# Eurex CCP LEIs
eurex_lei = ('529900LN3S50JPU47S06', '529900VIIHQECIS4ET47', '529900UT4DG0LG5R9O07')

In [ ]:
# one off: materialize the list of Eurex clearing members seen in sovereign futures trades
# (full scan of the REFIT table, run once; afterwards the main query only joins to the small table)
cm_table = 'xlab_ecb_prj_sftds_cb_common.hermesf_eurex_cm'

cursor.execute(f"DROP TABLE IF EXISTS {cm_table}")
cursor.execute(f"""
CREATE EXTERNAL TABLE {cm_table}
STORED AS PARQUET TBLPROPERTIES ('external.table.purge'='true')
AS
SELECT
    leg1_clearing_member_id AS lei,
    COUNT(*)              AS n_rows,
    MIN(reference_period) AS first_seen,
    MAX(reference_period) AS last_seen
FROM crp_emir_refit_ecb.emir_refit_ecb_trade_states_deduplicated
WHERE leg1_product_isin IN {euro_govs_isin}
    AND leg1_clearing_member_id IS NOT NULL
    AND leg1_clearing_member_id NOT IN {eurex_lei}
GROUP BY leg1_clearing_member_id
""")

In [ ]:
# check the clearing member list
df_cm = pd.read_sql_query(f"SELECT * FROM {cm_table} ORDER BY n_rows DESC", cnxn)
print(df_cm.shape)
df_cm

In [ ]:
# step 2: counterparty type (CCP / CM / CLIENT)
query = f"""
WITH base AS (
    SELECT
        e.tec_ruti,
        e.reference_period,
        e.leg1_direction,
        e.leg1_reporting_cpty_id,
        e.leg1_other_cpty_id,
        s_rep.sector AS rep_sector,
        s_oth.sector AS oth_sector,
        CASE WHEN e.leg1_reporting_cpty_id IN {eurex_lei} THEN 'CCP'
             WHEN cm_rep.lei IS NOT NULL                   THEN 'CM'
             ELSE 'CLIENT' END AS rep_type,
        CASE WHEN e.leg1_other_cpty_id IN {eurex_lei}     THEN 'CCP'
             WHEN cm_oth.lei IS NOT NULL                   THEN 'CM'
             ELSE 'CLIENT' END AS oth_type
    FROM crp_emir_refit_ecb.emir_refit_ecb_trade_states_deduplicated e
    LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_rep
        ON e.leg1_reporting_cpty_id = s_rep.lei
    LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_oth
        ON e.leg1_other_cpty_id = s_oth.lei
    LEFT JOIN {cm_table} cm_rep
        ON e.leg1_reporting_cpty_id = cm_rep.lei
    LEFT JOIN {cm_table} cm_oth
        ON e.leg1_other_cpty_id = cm_oth.lei
    WHERE e.reference_period = '{sample_date}'
        AND e.leg1_product_isin = '{sample_isin}'
        AND e.leg1_direction IN ('BYER', 'SLLR')
)
SELECT
    tec_ruti,
    reference_period,
    CASE WHEN leg1_direction = 'BYER' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS buyer_id,
    CASE WHEN leg1_direction = 'BYER' THEN rep_sector             ELSE oth_sector             END AS buyer_sector,
    CASE WHEN leg1_direction = 'BYER' THEN rep_type               ELSE oth_type               END AS buyer_type,
    CASE WHEN leg1_direction = 'SLLR' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS seller_id,
    CASE WHEN leg1_direction = 'SLLR' THEN rep_sector             ELSE oth_sector             END AS seller_sector,
    CASE WHEN leg1_direction = 'SLLR' THEN rep_type               ELSE oth_type               END AS seller_type
FROM base
"""

df = pd.read_sql_query(query, cnxn)
print(df.shape)
print(pd.crosstab(df['buyer_type'], df['seller_type']))
df.head()